#Naive Bayes


In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline


# ============================================
# STEMMING
# ============================================
def clean_text(text):
    """
    Membersihkan teks dengan:
    1. Konversi ke huruf kecil
    2. Menghapus angka
    3. Menghapus tanda baca
    4. Menghapus spasi berlebih
    """
    if not isinstance(text, str):
        return ""

    # 1. Konversi ke huruf kecil
    text = text.lower()

    # 2. Hapus angka
    text = re.sub(r'\d+', '', text)

    # 3. Hapus tanda baca
    text = re.sub(r'[^\w\s]', '', text)

    # 4. Hapus spasi berlebih
    text = ' '.join(text.split())

    return text


def display_label_mapping(encoder, label_name="Label"):
    """Menampilkan mapping label yang telah di-encode"""
    print(f"\n{label_name} Mappings:")
    for i, label in enumerate(encoder.classes_):
        print(f"  {label}: {i}")


# ============================================
# LOAD & INSPEKSI DATA
# ============================================
# Load dataset
print("=" * 50)
print("MEMUAT DATASET")
print("=" * 50)
df = pd.read_csv('250 Data_Manual.csv', delimiter=';')

# Display basic info
print("\n📊 INFORMASI DATASET:")
print("-" * 30)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

print("\n🔍 SAMPEL DATA (5 baris pertama):")
print("-" * 30)
print(df.head())

print("\n📋 INFORMASI KOLOM:")
print("-" * 30)
print(df.info())


# ============================================
# PREPROCESSING TEKS
# ============================================
print("\n" + "=" * 50)
print("PREPROCESSING TEKS")
print("=" * 50)

# Membersihkan teks
df['Ulasan_Bersih_Cleaned'] = df['Ulasan_Bersih'].apply(clean_text)

# Menampilkan contoh hasil cleaning
print("\n✅ HASIL CLEANING TEKS:")
print("-" * 30)
for i in range(min(3, len(df))):
    print(f"Original: {df['Ulasan_Bersih'].iloc[i][:50]}...")
    print(f"Cleaned : {df['Ulasan_Bersih_Cleaned'].iloc[i][:50]}...")
    print("-" * 20)


# ============================================
# ENCODING LABEL
# ============================================
print("\n" + "=" * 50)
print("ENCODING LABEL")
print("=" * 50)

# Membersihkan label (strip whitespace)
df['Labels'] = df['Labels'].str.strip()

# Cek label unik
print(f"\n🏷️  UNIQUE LABELS: {df['Labels'].unique()}")

# Encode labels
label_encoder = LabelEncoder()
df['Label_Encoded'] = label_encoder.fit_transform(df['Labels'])

# Tampilkan mapping
display_label_mapping(label_encoder, "LABEL")

# Cek distribusi label
print("\n📈 DISTRIBUSI LABEL:")
print("-" * 30)
label_counts = df['Labels'].value_counts()
for label, count in label_counts.items():
    print(f"{label}: {count} ({count/len(df)*100:.1f}%)")


# ============================================
# SPLIT DATA
# ============================================
print("\n" + "=" * 50)
print("SPLIT DATA TRAIN-TEST")
print("=" * 50)

# Split data
X = df['Ulasan_Bersih_Cleaned']
y = df['Label_Encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\n📊 DATA SPLITTING:")
print("-" * 30)
print(f"Total Data   : {len(df)}")
print(f"Train Data   : {len(X_train)} ({len(X_train)/len(df)*100:.1f}%)")
print(f"Test Data    : {len(X_test)} ({len(X_test)/len(df)*100:.1f}%)")
print(f"\nDistribusi Train: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Distribusi Test : {pd.Series(y_test).value_counts().to_dict()}")


# ============================================
# MODEL NAIVE BAYES
# ============================================
print("\n" + "=" * 50)
print("MODEL NAIVE BAYES")
print("=" * 50)

# Buat dan train model
nb_pipeline = make_pipeline(
    TfidfVectorizer(),
    MultinomialNB()
)

nb_pipeline.fit(X_train, y_train)
nb_predictions = nb_pipeline.predict(X_test)

# Evaluasi model
print("\n📋 CLASSIFICATION REPORT - NAIVE BAYES:")
print("-" * 50)
target_names = label_encoder.classes_
print(classification_report(
    y_test,
    nb_predictions,
    target_names=target_names,
    digits=4
))


MEMUAT DATASET

📊 INFORMASI DATASET:
------------------------------
Shape: (250, 2)
Columns: ['Ulasan_Bersih', 'Labels']

🔍 SAMPEL DATA (5 baris pertama):
------------------------------
                                       Ulasan_Bersih   Labels
0  kualitas gambar atur langsung kaya dulu penuru...  Negatif
1                                sangat bagus sekali  Positif
2                                              bagus  Positif
3                                            berguna  Positif
4  mending nonton steel bal run website baja dari...  Negatif

📋 INFORMASI KOLOM:
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Ulasan_Bersih  250 non-null    object
 1   Labels         250 non-null    object
dtypes: object(2)
memory usage: 4.0+ KB
None

PREPROCESSING TEKS

✅ HASIL CLEANING TEKS:
--------------------------

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#SVM

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC # Import Support Vector Classifier
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline


# PREPROCESSING TEKS (STEMMING/CLEANING)
# ============================================
def clean_text(text):
    """
    Membersihkan teks dengan:
    1. Konversi ke huruf kecil
    2. Menghapus angka
    3. Menghapus tanda baca
    4. Menghapus spasi berlebih
    """
    if not isinstance(text, str):
        return ""

    # 1. Konversi ke huruf kecil
    text = text.lower()

    # 2. Hapus angka
    text = re.sub(r'\d+', '', text)

    # 3. Hapus tanda baca
    text = re.sub(r'[^\w\s]', '', text)

    # 4. Hapus spasi berlebih
    text = ' '.join(text.split())

    return text


def display_label_mapping(encoder, label_name="Label"):
    """Menampilkan mapping label yang telah di-encode"""
    print(f"\n{label_name} Mappings:")
    for i, label in enumerate(encoder.classes_):
        print(f"  {label}: {i}")



# LOAD & INSPEKSI DATA
# ============================================
# Load dataset
print("=" * 50)
print("MEMUAT DATASET")
print("=" * 50)
df = pd.read_csv('250 Data_Manual.csv', delimiter=';')

# Display basic info
print("\n📊 INFORMASI DATASET:")
print("-" * 30)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

print("\n🔍 SAMPEL DATA (5 baris pertama):")
print("-" * 30)
print(df.head())

print("\n📋 INFORMASI KOLOM:")
print("-" * 30)
print(df.info())


# PREPROCESSING TEKS
# ============================================
print("\n" + "=" * 50)
print("PREPROCESSING TEKS")
print("=" * 50)

# Membersihkan teks
df['Ulasan_Bersih_Cleaned'] = df['Ulasan_Bersih'].apply(clean_text)

# Menampilkan contoh hasil cleaning
print("\n✅ HASIL CLEANING TEKS:")
print("-" * 30)
for i in range(min(3, len(df))):
    print(f"Original: {df['Ulasan_Bersih'].iloc[i][:50]}...")
    print(f"Cleaned : {df['Ulasan_Bersih_Cleaned'].iloc[i][:50]}...")
    print("-" * 20)



# ENCODING LABEL
# ============================================
print("\n" + "=" * 50)
print("ENCODING LABEL")
print("=" * 50)

# Membersihkan label (strip whitespace)
df['Labels'] = df['Labels'].str.strip()

# Encode labels
label_encoder = LabelEncoder()
df['Label_Encoded'] = label_encoder.fit_transform(df['Labels'])

# Tampilkan mapping
display_label_mapping(label_encoder, "LABEL")



# SPLIT DATA
# ============================================
print("\n" + "=" * 50)
print("SPLIT DATA TRAIN-TEST")
print("=" * 50)

# Split data
X = df['Ulasan_Bersih_Cleaned']
y = df['Label_Encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\n📊 DATA SPLITTING:")
print("-" * 30)
print(f"Total Data   : {len(df)}")
print(f"Train Data   : {len(X_train)} ({len(X_train)/len(df)*100:.1f}%)")
print(f"Test Data    : {len(X_test)} ({len(X_test)/len(df)*100:.1f}%)")
print(f"\nDistribusi Train: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Distribusi Test : {pd.Series(y_test).value_counts().to_dict()}")



# MODEL SUPPORT VECTOR MACHINE (SVM) dengan TF-IDF
# ============================================
print("\n" + "=" * 50)
print("MODEL SVM dengan TF-IDF")
print("=" * 50)

# Buat dan train model SVM
# Menggunakan kernel 'linear' yang efektif untuk klasifikasi teks
svm_pipeline = make_pipeline(
    TfidfVectorizer(), # Langkah 1: Ekstraksi fitur TF-IDF
    SVC(kernel='linear', random_state=42) # Langkah 2: Algoritma SVM
)

svm_pipeline.fit(X_train, y_train)
svm_predictions = svm_pipeline.predict(X_test)

# Evaluasi model
print("\n📋 CLASSIFICATION REPORT - SVM (TF-IDF):")
print("-" * 50)
target_names = label_encoder.classes_
print(classification_report(
    y_test,
    svm_predictions,
    target_names=target_names,
    digits=4
))

MEMUAT DATASET

📊 INFORMASI DATASET:
------------------------------
Shape: (250, 2)
Columns: ['Ulasan_Bersih', 'Labels']

🔍 SAMPEL DATA (5 baris pertama):
------------------------------
                                       Ulasan_Bersih   Labels
0  kualitas gambar atur langsung kaya dulu penuru...  Negatif
1                                sangat bagus sekali  Positif
2                                              bagus  Positif
3                                            berguna  Positif
4  mending nonton steel bal run website baja dari...  Negatif

📋 INFORMASI KOLOM:
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Ulasan_Bersih  250 non-null    object
 1   Labels         250 non-null    object
dtypes: object(2)
memory usage: 4.0+ KB
None

PREPROCESSING TEKS

✅ HASIL CLEANING TEKS:
--------------------------

#Logistic Regression

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression # Diperlukan untuk Logistic Regression
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline


# ============================================
# PREPROCESSING TEKS (STEMMING/CLEANING)
# ============================================
def clean_text(text):
    """
    Membersihkan teks dengan:
    1. Konversi ke huruf kecil
    2. Menghapus angka
    3. Menghapus tanda baca
    4. Menghapus spasi berlebih
    """
    if not isinstance(text, str):
        return ""

    # 1. Konversi ke huruf kecil
    text = text.lower()

    # 2. Hapus angka
    text = re.sub(r'\d+', '', text)

    # 3. Hapus tanda baca
    text = re.sub(r'[^\w\s]', '', text)

    # 4. Hapus spasi berlebih
    text = ' '.join(text.split())

    return text


def display_label_mapping(encoder, label_name="Label"):
    """Menampilkan mapping label yang telah di-encode"""
    print(f"\n{label_name} Mappings:")
    for i, label in enumerate(encoder.classes_):
        print(f"  {label}: {i}")


# ============================================
# LOAD & INSPEKSI DATA
# ============================================
# Load dataset
print("=" * 50)
print("MEMUAT DATASET")
print("=" * 50)
df = pd.read_csv('250 Data_Manual.csv', delimiter=';')

# Display basic info
print("\n📊 INFORMASI DATASET:")
print("-" * 30)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

print("\n🔍 SAMPEL DATA (5 baris pertama):")
print("-" * 30)
print(df.head())

print("\n📋 INFORMASI KOLOM:")
print("-" * 30)
print(df.info())

# ============================================
# PREPROCESSING TEKS
# ============================================
print("\n" + "=" * 50)
print("PREPROCESSING TEKS")
print("=" * 50)

# Membersihkan teks
df['Ulasan_Bersih_Cleaned'] = df['Ulasan_Bersih'].apply(clean_text)

# Menampilkan contoh hasil cleaning
print("\n✅ HASIL CLEANING TEKS:")
print("-" * 30)
for i in range(min(3, len(df))):
    print(f"Original: {df['Ulasan_Bersih'].iloc[i][:50]}...")
    print(f"Cleaned : {df['Ulasan_Bersih_Cleaned'].iloc[i][:50]}...")
    print("-" * 20)


# ============================================
# ENCODING LABEL
# ============================================
print("\n" + "=" * 50)
print("ENCODING LABEL")
print("=" * 50)

# Membersihkan label (strip whitespace)
df['Labels'] = df['Labels'].str.strip()

# Encode labels
label_encoder = LabelEncoder()
df['Label_Encoded'] = label_encoder.fit_transform(df['Labels'])

# Tampilkan mapping
display_label_mapping(label_encoder, "LABEL")


# ============================================
# SPLIT DATA
# ============================================
print("\n" + "=" * 50)
print("SPLIT DATA TRAIN-TEST")
print("=" * 50)

X = df['Ulasan_Bersih_Cleaned']
y = df['Label_Encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\n📊 DATA SPLITTING:")
print("-" * 30)
print(f"Total Data   : {len(df)}")
print(f"Train Data   : {len(X_train)} ({len(X_train)/len(df)*100:.1f}%)")
print(f"Test Data    : {len(X_test)} ({len(X_test)/len(df)*100:.1f}%)")
print(f"\nDistribusi Train: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Distribusi Test : {pd.Series(y_test).value_counts().to_dict()}")


# ============================================
# MODEL LOGISTIC REGRESSION dengan TF-IDF
# ============================================
print("\n" + "=" * 50)
print("MODEL LOGISTIC REGRESSION")
print("=" * 50)

# Buat dan train model
# Logistic Regression adalah model linear yang baik untuk klasifikasi teks
lr_pipeline = make_pipeline(
    TfidfVectorizer(),
    # C=1.0 adalah nilai default, multi_class='auto' menangani klasifikasi multi-kelas
    LogisticRegression(random_state=42, max_iter=1000)
)
lr_pipeline.fit(X_train, y_train)
lr_predictions = lr_pipeline.predict(X_test)

# Evaluasi model
print("\n📋 CLASSIFICATION REPORT - LOGISTIC REGRESSION:")
print("-" * 50)
target_names = label_encoder.classes_
print(classification_report(
    y_test,
    lr_predictions,
    target_names=target_names,
    digits=4
))

MEMUAT DATASET

📊 INFORMASI DATASET:
------------------------------
Shape: (250, 2)
Columns: ['Ulasan_Bersih', 'Labels']

🔍 SAMPEL DATA (5 baris pertama):
------------------------------
                                       Ulasan_Bersih   Labels
0  kualitas gambar atur langsung kaya dulu penuru...  Negatif
1                                sangat bagus sekali  Positif
2                                              bagus  Positif
3                                            berguna  Positif
4  mending nonton steel bal run website baja dari...  Negatif

📋 INFORMASI KOLOM:
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Ulasan_Bersih  250 non-null    object
 1   Labels         250 non-null    object
dtypes: object(2)
memory usage: 4.0+ KB
None

PREPROCESSING TEKS

✅ HASIL CLEANING TEKS:
--------------------------

# Random Forest

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier # Diperlukan untuk Random Forest
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline


# ============================================
# PREPROCESSING TEKS (STEMMING/CLEANING)
# ============================================
def clean_text(text):
    """
    Membersihkan teks dengan:
    1. Konversi ke huruf kecil
    2. Menghapus angka
    3. Menghapus tanda baca
    4. Menghapus spasi berlebih
    """
    if not isinstance(text, str):
        return ""

    # 1. Konversi ke huruf kecil
    text = text.lower()

    # 2. Hapus angka
    text = re.sub(r'\d+', '', text)

    # 3. Hapus tanda baca
    text = re.sub(r'[^\w\s]', '', text)

    # 4. Hapus spasi berlebih
    text = ' '.join(text.split())

    return text


def display_label_mapping(encoder, label_name="Label"):
    """Menampilkan mapping label yang telah di-encode"""
    print(f"\n{label_name} Mappings:")
    for i, label in enumerate(encoder.classes_):
        print(f"  {label}: {i}")


# ============================================
# LOAD & INSPEKSI DATA
# ============================================
# Load dataset
print("=" * 50)
print("MEMUAT DATASET")
print("=" * 50)
df = pd.read_csv('250 Data_Manual.csv', delimiter=';')

# Display basic info
print("\n📊 INFORMASI DATASET:")
print("-" * 30)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

print("\n🔍 SAMPEL DATA (5 baris pertama):")
print("-" * 30)
print(df.head())

print("\n📋 INFORMASI KOLOM:")
print("-" * 30)
print(df.info())

# ============================================
# PREPROCESSING TEKS
# ============================================
print("\n" + "=" * 50)
print("PREPROCESSING TEKS")
print("=" * 50)

# Membersihkan teks
df['Ulasan_Bersih_Cleaned'] = df['Ulasan_Bersih'].apply(clean_text)

# Menampilkan contoh hasil cleaning
print("\n✅ HASIL CLEANING TEKS:")
print("-" * 30)
for i in range(min(3, len(df))):
    print(f"Original: {df['Ulasan_Bersih'].iloc[i][:50]}...")
    print(f"Cleaned : {df['Ulasan_Bersih_Cleaned'].iloc[i][:50]}...")
    print("-" * 20)


# ============================================
# ENCODING LABEL
# ============================================
print("\n" + "=" * 50)
print("ENCODING LABEL")
print("=" * 50)

# Membersihkan label (strip whitespace)
df['Labels'] = df['Labels'].str.strip()

# Encode labels
label_encoder = LabelEncoder()
df['Label_Encoded'] = label_encoder.fit_transform(df['Labels'])

# Tampilkan mapping
display_label_mapping(label_encoder, "LABEL")


# ============================================
# SPLIT DATA
# ============================================
print("\n" + "=" * 50)
print("SPLIT DATA TRAIN-TEST")
print("=" * 50)

# Split data
X = df['Ulasan_Bersih_Cleaned']
y = df['Label_Encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\n📊 DATA SPLITTING:")
print("-" * 30)
print(f"Total Data   : {len(df)}")
print(f"Train Data   : {len(X_train)} ({len(X_train)/len(df)*100:.1f}%)")
print(f"Test Data    : {len(X_test)} ({len(X_test)/len(df)*100:.1f}%)")
print(f"\nDistribusi Train: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Distribusi Test : {pd.Series(y_test).value_counts().to_dict()}")


# ============================================
# MODEL RANDOM FOREST dengan TF-IDF
# ============================================
print("\n" + "=" * 50)
print("MODEL RANDOM FOREST")
print("=" * 50)

# Buat dan train model
# Random Forest adalah model berbasis ensemble
rf_pipeline = make_pipeline(
    TfidfVectorizer(),
    # Menggunakan parameter default dengan random_state untuk reproduksibilitas
    RandomForestClassifier(random_state=42)
)

rf_pipeline.fit(X_train, y_train)
rf_predictions = rf_pipeline.predict(X_test)

# Evaluasi model
print("\n📋 CLASSIFICATION REPORT - RANDOM FOREST:")
print("-" * 50)
target_names = label_encoder.classes_
print(classification_report(
    y_test,
    rf_predictions,
    target_names=target_names,
    digits=4
))

MEMUAT DATASET

📊 INFORMASI DATASET:
------------------------------
Shape: (250, 2)
Columns: ['Ulasan_Bersih', 'Labels']

🔍 SAMPEL DATA (5 baris pertama):
------------------------------
                                       Ulasan_Bersih   Labels
0  kualitas gambar atur langsung kaya dulu penuru...  Negatif
1                                sangat bagus sekali  Positif
2                                              bagus  Positif
3                                            berguna  Positif
4  mending nonton steel bal run website baja dari...  Negatif

📋 INFORMASI KOLOM:
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Ulasan_Bersih  250 non-null    object
 1   Labels         250 non-null    object
dtypes: object(2)
memory usage: 4.0+ KB
None

PREPROCESSING TEKS

✅ HASIL CLEANING TEKS:
--------------------------